# Rationale Explanations Visualization

This notebook visualizes pre-computed LLM-generated rationales.

Rationales are short text justifications produced by a local LLM (e.g. Llama-3.2-3B)
explaining why the task model's prediction makes sense for a given sample.

Cache location: `data/{model_dir}/rationales/{llm_model_slug}.jsonl`

## Parameters

In [ ]:
# ---------------------------------------------------------------------------
# Configuration: adjust these to select the dataset / LLM model
# ---------------------------------------------------------------------------

# Dataset short name: "RT", "GE", "BIOS", "AG", "IMDB", "E", "HE"
DATASET_ABBREV = "RT"

# LLM model used for rationale generation (short name or full HF path)
# Common values: "llama3.2-3b", "qwen3.5-9b"
LLM_MODEL = "llama3.2-3b"

# Classes subset index (within DATASET_CLASSES_SUBSETS for the dataset)
CLASSES_SUBSET_IDX = 0

# Seed for sample selection
SEED = 0

# Number of samples per seed
NB_SAMPLES = 5

## Imports and path resolution

In [ ]:
import sys
from pathlib import Path

# Add repo root to path so we can import utils
REPO_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(REPO_ROOT))

import json
import torch

from utils.data import (
    MODELS_DATASETS,
    ABBREVIATIONS,
    DATASET_CLASSES_NAMES,
    DATASET_CLASSES_SUBSETS,
    LLM_MODELS,
    get_save_root,
    iter_jsonl,
    resolve_llm_model,
)

In [ ]:
# Resolve full dataset/model names from abbreviation
dataset_name = next(k for k, v in ABBREVIATIONS["datasets"].items() if v == DATASET_ABBREV)
model_name = next(k for k, v in MODELS_DATASETS.items() if v == dataset_name)

# Classes subset
classes_subset = DATASET_CLASSES_SUBSETS[dataset_name][CLASSES_SUBSET_IDX]
classes_names = [DATASET_CLASSES_NAMES[dataset_name][i] for i in classes_subset]

# Resolve LLM model path
llm_model_full = resolve_llm_model(LLM_MODEL)
llm_model_slug = llm_model_full.replace("/", "_")

# Paths
save_root = REPO_ROOT / get_save_root(model_name)
rationale_path = save_root / "rationales" / f"{llm_model_slug}.jsonl"

print(f"Dataset: {dataset_name}")
print(f"Model: {model_name}")
print(f"LLM for rationales: {llm_model_full}")
print(f"Classes: {classes_names} (indices {classes_subset})")
print(f"Rationale file: {rationale_path}")
print(f"Exists: {rationale_path.exists()}")

## Load rationales

In [ ]:
# Load all rationales from the JSONL cache
rationales: dict[int, str] = {}
if rationale_path.exists():
    for record in iter_jsonl(rationale_path):
        rationales[record["sample_id"]] = record["rationale"]
    print(f"Loaded {len(rationales)} rationales")
else:
    print(f"WARNING: Rationale file not found at {rationale_path}")
    print("Run `python scripts/make_prompts.py rationales {dataset} --llm-model {model}` to generate them.")

## Load sample selection

In [ ]:
# Load sample selection for this seed
classes_str = "-".join(str(c) for c in classes_subset)
local_elements_path = save_root / f"local_elements_classes_{classes_str}_n{NB_SAMPLES}.json"

if local_elements_path.exists():
    with open(local_elements_path) as f:
        local_elements = json.load(f)
    seed_data = local_elements[str(SEED)]
    sample_indices = seed_data["indices"]
    sample_texts = seed_data["texts"]
    sample_predictions = seed_data["predictions"]
    sample_labels = seed_data["labels"]
    print(f"Seed {SEED}: {len(sample_indices)} samples")
else:
    print(f"WARNING: Local elements file not found: {local_elements_path}")
    # Fallback: show rationales for the first N available sample_ids
    sample_indices = sorted(rationales.keys())[:NB_SAMPLES]
    sample_texts = None
    sample_predictions = None
    sample_labels = None
    print(f"Falling back to first {len(sample_indices)} rationale sample_ids")

## Display rationales

In [ ]:
from IPython.display import display, HTML

html_parts = []
for i, idx in enumerate(sample_indices):
    text = sample_texts[i] if sample_texts else f"(sample_id={idx})"
    pred = sample_predictions[i] if sample_predictions else "?"
    pred_name = DATASET_CLASSES_NAMES[dataset_name][pred] if isinstance(pred, int) else str(pred)
    label = sample_labels[i] if sample_labels else "?"
    label_name = DATASET_CLASSES_NAMES[dataset_name][label] if isinstance(label, int) else str(label)
    rationale = rationales.get(idx, "<not computed>")

    html_parts.append(f"""
    <div style="border:1px solid #ccc; padding:10px; margin:8px 0; border-radius:5px;">
        <b>Sample {i}</b> (idx={idx}) &mdash;
        Label: <span style="color:green">{label_name}</span>,
        Prediction: <span style="color:blue">{pred_name}</span>
        <br><br>
        <em>"{text[:200]}{'...' if len(text) > 200 else ''}"</em>
        <br><br>
        <b>Rationale:</b> {rationale}
    </div>
    """)

display(HTML("".join(html_parts)))

## Rationale statistics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if rationales:
    # Word count distribution
    word_counts = [len(r.split()) for r in rationales.values()]
    char_counts = [len(r) for r in rationales.values()]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(word_counts, bins=30, edgecolor="black", alpha=0.7)
    axes[0].set_xlabel("Word count")
    axes[0].set_ylabel("Frequency")
    axes[0].set_title(f"Rationale word count distribution (n={len(rationales)})")
    axes[0].axvline(np.mean(word_counts), color="red", linestyle="--", label=f"Mean: {np.mean(word_counts):.1f}")
    axes[0].legend()

    axes[1].hist(char_counts, bins=30, edgecolor="black", alpha=0.7, color="orange")
    axes[1].set_xlabel("Character count")
    axes[1].set_ylabel("Frequency")
    axes[1].set_title(f"Rationale character count distribution (n={len(rationales)})")
    axes[1].axvline(np.mean(char_counts), color="red", linestyle="--", label=f"Mean: {np.mean(char_counts):.1f}")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    print(f"\nTotal rationales: {len(rationales)}")
    print(f"Word count — min: {min(word_counts)}, max: {max(word_counts)}, mean: {np.mean(word_counts):.1f}")
    print(f"Char count — min: {min(char_counts)}, max: {max(char_counts)}, mean: {np.mean(char_counts):.1f}")
else:
    print("No rationales loaded — skipping statistics.")

## View generated prompts (from prompt JSONL)

In [ ]:
# Optionally inspect the rationale-based simulatability prompts
prompt_file = REPO_ROOT / "data" / "prompts" / f"{DATASET_ABBREV}_rationales.jsonl"

if prompt_file.exists():
    prompts = list(iter_jsonl(prompt_file))
    print(f"Loaded {len(prompts)} prompt entries from {prompt_file.name}")

    # Show the first entry with rationale content (prompt_type R1)
    for entry in prompts:
        key = entry["key"]
        if "R1" in key:
            print(f"\nKey: {key}")
            print(f"\n--- System prompt (first 500 chars) ---")
            print(entry["system_prompt"][:500])
            print(f"\n--- First user prompt (first 300 chars) ---")
            print(entry["user_prompts"][0][:300])
            print(f"\n--- Expected answers ---")
            print(entry["expected_answers"])
            break
else:
    print(f"Prompt file not found: {prompt_file}")
    print("Run `python scripts/make_prompts.py rationales {dataset}` to generate prompts.")